# Notebook 04 - Observation Modes

## Purpose

This notebook investigates whether ALMA Archive metadata can support a
traceable classification of reconstructed spectral setups as continuum,
spectral-line FDM, mixed, or undetermined.

The notebook does not implement a formal duplication decision. It tests
whether the Archive contains the metadata required by Appendix A of the
ALMA Users' Policies.

## Authoritative Sources

1. Supervisor project specification: `Internship: Duplication Check Tool`,
   Nordic ARC node, August 2026.
2. ALMA Duplicate Observations:
   https://almascience.eso.org/proposing/duplications
3. ALMA Users' Policies, Appendix A:
   https://almascience.eso.org/documents-and-tools/latest/alma-user-policies
4. ALMA Science Archive Notebooks:
   https://almascience.eso.org/alma-data/archive/archive-notebooks/

## Scope Boundary

This notebook investigates evidence and field semantics. It does not:

- decide whether an observation is formally duplicated;
- introduce frequency or sensitivity tolerances;
- assume that Archive row bandwidth equals correlator-window bandwidth;
- infer FDM solely from an unvalidated resolution threshold;
- implement the HTML interface.

## Current Stage

This version completes the corrected schema inventory and stratified sample
construction. It ensures that both supervisor projects are represented,
includes all available mode-related Archive fields, and verifies complete
Member OUS retrieval before later structure and mode experiments.

## Notebook Roadmap

1. Environment and reproducibility
2. TAP schema and policy-to-field inventory
3. Corrected stratified sample construction
4. Complete Member OUS retrieval
5. Source-SPW reconstruction (next stage)
6. Continuum evidence (next stage)
7. FDM and sensitivity evidence (next stage)
8. Supervisor retrieval cases (next stage)
9. Final conclusions and open questions (next stage)

## 1. Environment and Reproducibility

The notebook imports the reusable production frequency-support parser. No
parser implementation is copied from Notebook 03.

In [20]:
from __future__ import annotations

from datetime import datetime, timezone
import platform
import sys

import astropy
from astropy import units as u
import pandas as pd
import pyvo

from alma_duplicate.parsers.frequency_support import (
    parse_frequency_support,
)


print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Astropy:", astropy.__version__)
print("Pandas:", pd.__version__)
print("PyVO:", pyvo.__version__)
print("Run time UTC:", datetime.now(timezone.utc).isoformat())

Python: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 08:06:15) [Clang 14.0.6 ]
Executable: /Users/nana/opt/anaconda3/envs/alma-duplication/bin/python
Platform: macOS-26.4.1-x86_64-i386-64bit
Astropy: 8.0.1
Pandas: 3.0.5
PyVO: 1.9.1
Run time UTC: 2026-08-24T12:06:31.946824+00:00


In [21]:
TAP_URL = "https://almascience.eso.org/tap"

tap_service = pyvo.dal.TAPService(TAP_URL)

print("TAP endpoint:", TAP_URL)
print("Service:", tap_service)

TAP endpoint: https://almascience.eso.org/tap
Service: TAPService(baseurl : 'https://almascience.eso.org/tap', description : 'None')


In [22]:
def run_tap_query(
    adql: str,
    *,
    maxrec: int,
    label: str,
):
    '''Run one TAP query while recording its execution context.'''

    print(f"\n--- {label} ---")
    print("Endpoint:", TAP_URL)
    print("MAXREC:", maxrec)
    print("ADQL:")
    print(adql)

    result = tap_service.search(
        adql,
        maxrec=maxrec,
    )
    table = result.to_table()

    print("Retrieved rows:", len(table))

    return table


def quote_adql_string(value: str) -> str:
    '''Quote one trusted identifier for an ADQL string literal.'''

    escaped_value = value.replace("'", "''")
    return f"'{escaped_value}'"

## 2. TAP Schema and Policy-to-Field Inventory

This section determines which policy-relevant concepts are represented
directly by Archive columns and which require parsing, reconstruction, or
scientific interpretation.

Field names alone are not treated as sufficient evidence. Units,
descriptions, missingness, cardinality, and ownership must also be tested.

In [23]:
schema_query = '''
SELECT
    column_name,
    datatype,
    unit,
    ucd,
    description
FROM TAP_SCHEMA.columns
WHERE table_name = 'ivoa.obscore'
ORDER BY column_name
'''

schema_table = run_tap_query(
    schema_query,
    maxrec=5000,
    label="Retrieve ivoa.obscore schema",
)

schema_df = schema_table.to_pandas()

print("Archive column count:", len(schema_df))
display(schema_df.head())


--- Retrieve ivoa.obscore schema ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 5000
ADQL:

SELECT
    column_name,
    datatype,
    unit,
    ucd,
    description
FROM TAP_SCHEMA.columns
WHERE table_name = 'ivoa.obscore'
ORDER BY column_name

Retrieved rows: 73
Archive column count: 73


,column_name,datatype,unit,ucd,description
0,access_estsize,int,kbyte,phys.size;meta.file,Estimated size of datasets in kilobytes
1,access_format,char,,meta.code.mime,Content format of the data
2,access_url,char,,meta.ref.url,URL to download the data
3,antenna_arrays,char,,meta.code.member;instr.setup,"Blank-separated list of Pad:Antenna pairs, i.e..."
4,asdm_uid,char,,meta.id,UID of the ASDM containing this Field.


In [24]:
available_columns = set(
    schema_df["column_name"]
    .astype("string")
    .dropna()
)

core_required_columns = {
    "proposal_id",
    "group_ous_uid",
    "member_ous_uid",
    "obs_id",
    "asdm_uid",
    "target_name",
    "s_ra",
    "s_dec",
    "s_region",
    "frequency",
    "bandwidth",
    "frequency_support",
    "spatial_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
    "antenna_arrays",
    "is_mosaic",
}

missing_core_columns = (
    core_required_columns - available_columns
)

print("Required core columns:", len(core_required_columns))
print("Missing core columns:", sorted(missing_core_columns))

if missing_core_columns:
    raise RuntimeError(
        "Required Archive columns are missing: "
        f"{sorted(missing_core_columns)}"
    )

Required core columns: 17
Missing core columns: []


In [25]:
extended_mode_pattern = (
    r"mode|correl|fdm|tdm|spectral|velocity|"
    r"resolution|polar|pol_|band|scan|intent|array"
)

extended_mode_columns_df = schema_df[
    schema_df["column_name"]
    .astype("string")
    .str.contains(
        extended_mode_pattern,
        case=False,
        regex=True,
        na=False,
    )
].copy()

display(
    extended_mode_columns_df[
        [
            "column_name",
            "datatype",
            "unit",
            "ucd",
            "description",
        ]
    ]
)

,column_name,datatype,unit,ucd,description
3,antenna_arrays,char,,meta.code.member;instr.setup,"Blank-separated list of Pad:Antenna pairs, i.e..."
6,band_list,char,,,Space delimited list of bands
7,bandwidth,double,Hz,em.freq;instr.bandpass,Total Bandwidth
11,cont_sensitivity_bandwidth,double,mJy/beam,,Estimated noise in the aggregated continuum ba...
17,em_resolution,double,m,spect.resolution;stat.mean,Estimated frequency resolution from all the sp...
39,pol_states,char,,meta.code;phys.polarization,polarization states present in the data
40,pol_xel,int,,meta.number,Number of polarization samples
53,s_resolution,double,arcsec,pos.angResolution,typical spatial resolution
56,scan_intent,char,,meta.code.class;obs,Scan intent list for the observed field.
62,spatial_resolution,double,arcsec,,Average of the maximum and minimum spatial res...


In [26]:
desired_archive_columns = [
    "proposal_id",
    "group_ous_uid",
    "member_ous_uid",
    "obs_id",
    "asdm_uid",
    "target_name",
    "s_ra",
    "s_dec",
    "s_region",
    "frequency",
    "bandwidth",
    "frequency_support",
    "spectral_resolution",
    "velocity_resolution",
    "em_resolution",
    "s_resolution",
    "spatial_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
    "antenna_arrays",
    "is_mosaic",
    "pol_states",
    "band_list",
    "scan_intent",
]

selected_archive_columns = [
    column_name
    for column_name in desired_archive_columns
    if column_name in available_columns
]

missing_optional_columns = sorted(
    set(desired_archive_columns)
    - set(selected_archive_columns)
)

print("Selected Archive columns:")
for column_name in selected_archive_columns:
    print(" -", column_name)

print("\nUnavailable optional columns:")
for column_name in missing_optional_columns:
    print(" -", column_name)

if "spectral_resolution" not in selected_archive_columns:
    raise RuntimeError(
        "spectral_resolution is required for the next experiment."
    )

Selected Archive columns:
 - proposal_id
 - group_ous_uid
 - member_ous_uid
 - obs_id
 - asdm_uid
 - target_name
 - s_ra
 - s_dec
 - s_region
 - frequency
 - bandwidth
 - frequency_support
 - spectral_resolution
 - velocity_resolution
 - em_resolution
 - s_resolution
 - spatial_resolution
 - sensitivity_10kms
 - cont_sensitivity_bandwidth
 - antenna_arrays
 - is_mosaic
 - pol_states
 - band_list
 - scan_intent

Unavailable optional columns:


In [27]:
selected_schema_df = (
    schema_df[
        schema_df["column_name"].isin(
            selected_archive_columns
        )
    ]
    [
        [
            "column_name",
            "datatype",
            "unit",
            "ucd",
            "description",
        ]
    ]
    .sort_values("column_name")
    .reset_index(drop=True)
)

archive_unit_by_column = (
    selected_schema_df
    .set_index("column_name")["unit"]
    .to_dict()
)

display(selected_schema_df)

for field_name in [
    "frequency",
    "bandwidth",
    "spectral_resolution",
    "velocity_resolution",
    "spatial_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
]:
    if field_name in archive_unit_by_column:
        print(
            field_name,
            "->",
            archive_unit_by_column[field_name],
        )

,column_name,datatype,unit,ucd,description
0,antenna_arrays,char,,meta.code.member;instr.setup,"Blank-separated list of Pad:Antenna pairs, i.e..."
1,asdm_uid,char,,meta.id,UID of the ASDM containing this Field.
2,band_list,char,,,Space delimited list of bands
3,bandwidth,double,Hz,em.freq;instr.bandpass,Total Bandwidth
4,cont_sensitivity_bandwidth,double,mJy/beam,,Estimated noise in the aggregated continuum ba...
5,em_resolution,double,m,spect.resolution;stat.mean,Estimated frequency resolution from all the sp...
6,frequency,double,GHz,em.freq;obs;meta.main,Observed (tuned) reference frequency on the sky.
7,frequency_support,char,GHz,em.freq;obs;meta.main,All frequency ranges used by the field
8,group_ous_uid,char,,,Group OUS ID
9,is_mosaic,char,,,Flag to indicate if this ASDM represents a mos...


frequency -> GHz
bandwidth -> Hz
spectral_resolution -> kHz
velocity_resolution -> m/s
spatial_resolution -> arcsec
sensitivity_10kms -> mJy/beam
cont_sensitivity_bandwidth -> mJy/beam


In [28]:
policy_field_questions = pd.DataFrame(
    [
        {
            "policy_concept": "Correlator window",
            "candidate_archive_fields": (
                "frequency_support, obs_id, frequency, bandwidth"
            ),
            "required_experiment": (
                "Compare component counts, SPW identifiers, and widths"
            ),
        },
        {
            "policy_concept": "Continuum window bandwidth > 1.8 GHz",
            "candidate_archive_fields": (
                "bandwidth, parsed frequency-support interval width"
            ),
            "required_experiment": (
                "Test threshold agreement and quantify discrepancies"
            ),
        },
        {
            "policy_concept": "FDM mode",
            "candidate_archive_fields": "No direct field confirmed",
            "required_experiment": (
                "Inspect direct fields and raw representations; do not infer "
                "FDM from an unvalidated resolution threshold"
            ),
        },
        {
            "policy_concept": "Per-channel sensitivity",
            "candidate_archive_fields": (
                "frequency_support @native, sensitivity_10kms, "
                "spectral_resolution"
            ),
            "required_experiment": (
                "Determine basis, ownership, and smoothing readiness"
            ),
        },
        {
            "policy_concept": "Aggregate continuum sensitivity",
            "candidate_archive_fields": "cont_sensitivity_bandwidth",
            "required_experiment": (
                "Determine ownership and relation to the complete setup"
            ),
        },
        {
            "policy_concept": "Mosaic pointing overlap",
            "candidate_archive_fields": (
                "is_mosaic, s_region, s_ra, s_dec"
            ),
            "required_experiment": (
                "Determine whether pointings are individually reconstructable"
            ),
        },
    ]
)

display(policy_field_questions)

,policy_concept,candidate_archive_fields,required_experiment
0,Correlator window,"frequency_support, obs_id, frequency, bandwidth","Compare component counts, SPW identifiers, and..."
1,Continuum window bandwidth > 1.8 GHz,"bandwidth, parsed frequency-support interval w...",Test threshold agreement and quantify discrepa...
2,FDM mode,No direct field confirmed,Inspect direct fields and raw representations;...
3,Per-channel sensitivity,"frequency_support @native, sensitivity_10kms, ...","Determine basis, ownership, and smoothing read..."
4,Aggregate continuum sensitivity,cont_sensitivity_bandwidth,Determine ownership and relation to the comple...
5,Mosaic pointing overlap,"is_mosaic, s_region, s_ra, s_dec",Determine whether pointings are individually r...


## 3. Corrected Stratified Sample Construction

Candidate queries are used only to discover Member OUS identifiers. Complete
Member OUS datasets are counted and retrieved in the next section.

The sample is purposive rather than statistically representative. It covers:

- rows above the 1.8 GHz continuum-window boundary;
- narrow-bandwidth rows;
- mosaic observations;
- both supervisor-provided positive-retrieval projects.

The supervisor projects are sampled separately by `proposal_id`. This avoids
the earlier error in which a shared `supervisor_projects` stratum selected only
the first project before applying the six-member limit.

In [29]:
bandwidth_unit_text = archive_unit_by_column["bandwidth"]
bandwidth_archive_unit = u.Unit(bandwidth_unit_text)

continuum_threshold_archive = (
    1.8 * u.GHz
).to_value(bandwidth_archive_unit)

print(
    "1.8 GHz in Archive bandwidth unit:",
    continuum_threshold_archive,
    bandwidth_archive_unit,
)

1.8 GHz in Archive bandwidth unit: 1800000000.0 Hz


In [30]:
discovery_queries = {
    "wide_bandwidth": f'''
        SELECT TOP 100
            proposal_id,
            member_ous_uid,
            obs_id,
            bandwidth,
            frequency,
            is_mosaic
        FROM ivoa.obscore
        WHERE science_observation = 'T'
        AND bandwidth > {continuum_threshold_archive}
        AND member_ous_uid IS NOT NULL
    ''',
    "narrow_bandwidth": f'''
        SELECT TOP 100
            proposal_id,
            member_ous_uid,
            obs_id,
            bandwidth,
            frequency,
            is_mosaic
        FROM ivoa.obscore
        WHERE science_observation = 'T'
        AND bandwidth < {continuum_threshold_archive / 4}
        AND member_ous_uid IS NOT NULL
    ''',
    "mosaic": '''
        SELECT TOP 100
            proposal_id,
            member_ous_uid,
            obs_id,
            bandwidth,
            frequency,
            is_mosaic
        FROM ivoa.obscore
        WHERE science_observation = 'T'
        AND is_mosaic = 'T'
        AND member_ous_uid IS NOT NULL
    ''',
    "supervisor_projects": '''
        SELECT TOP 500
            proposal_id,
            member_ous_uid,
            obs_id,
            bandwidth,
            frequency,
            is_mosaic
        FROM ivoa.obscore
        WHERE science_observation = 'T'
        AND proposal_id IN (
            '2021.A.00028.S',
            '2018.1.00294.S'
        )
        AND member_ous_uid IS NOT NULL
    ''',
}

In [31]:
discovery_frames = []

for stratum_name, query in discovery_queries.items():
    table = run_tap_query(
        query,
        maxrec=500,
        label=f"Discover {stratum_name}",
    )

    frame = table.to_pandas()
    frame["sample_stratum"] = stratum_name
    discovery_frames.append(frame)

discovery_df = pd.concat(
    discovery_frames,
    ignore_index=True,
)

print("Discovery rows:", len(discovery_df))
display(discovery_df.head())


--- Discover wide_bandwidth ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 500
ADQL:

        SELECT TOP 100
            proposal_id,
            member_ous_uid,
            obs_id,
            bandwidth,
            frequency,
            is_mosaic
        FROM ivoa.obscore
        WHERE science_observation = 'T'
        AND bandwidth > 1800000000.0
        AND member_ous_uid IS NOT NULL
    
Retrieved rows: 100

--- Discover narrow_bandwidth ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 500
ADQL:

        SELECT TOP 100
            proposal_id,
            member_ous_uid,
            obs_id,
            bandwidth,
            frequency,
            is_mosaic
        FROM ivoa.obscore
        WHERE science_observation = 'T'
        AND bandwidth < 450000000.0
        AND member_ous_uid IS NOT NULL
    
Retrieved rows: 100

--- Discover mosaic ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 500
ADQL:

        SELECT TOP 100
            proposal_id,
            mem

,proposal_id,member_ous_uid,obs_id,bandwidth,frequency,is_mosaic,sample_stratum
0,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.19,2.000000e+09,225.732519,F,wide_bandwidth
1,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.21,2.000000e+09,237.290863,F,wide_bandwidth
2,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.23,2.000000e+09,240.296886,F,wide_bandwidth
3,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.25,1.875000e+09,223.836776,F,wide_bandwidth
4,2016.1.00447.S,uid://A001/X88f/X28a,uid://A001/X88f/X28a.source.sigOri_583.spw.23,2.000000e+09,233.998205,F,wide_bandwidth


In [32]:
regular_manifest_df = (
    discovery_df[
        discovery_df["sample_stratum"]
        != "supervisor_projects"
    ]
    [
        [
            "sample_stratum",
            "proposal_id",
            "member_ous_uid",
        ]
    ]
    .dropna(subset=["member_ous_uid"])
    .drop_duplicates()
    .groupby(
        "sample_stratum",
        group_keys=False,
    )
    .head(6)
)

supervisor_manifest_df = (
    discovery_df[
        discovery_df["sample_stratum"]
        == "supervisor_projects"
    ]
    [
        [
            "sample_stratum",
            "proposal_id",
            "member_ous_uid",
        ]
    ]
    .dropna(subset=["member_ous_uid"])
    .drop_duplicates()
    .groupby(
        "proposal_id",
        group_keys=False,
    )
    .head(2)
)

sample_manifest_df = (
    pd.concat(
        [
            regular_manifest_df,
            supervisor_manifest_df,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["member_ous_uid"]
    )
    .reset_index(drop=True)
)

expected_supervisor_projects = {
    "2021.A.00028.S",
    "2018.1.00294.S",
}

selected_supervisor_projects = set(
    sample_manifest_df.loc[
        sample_manifest_df["sample_stratum"]
        .eq("supervisor_projects"),
        "proposal_id",
    ]
)

print("Sample manifest:")
display(sample_manifest_df)

print(
    "Selected supervisor projects:",
    selected_supervisor_projects,
)

missing_supervisor_projects = (
    expected_supervisor_projects
    - selected_supervisor_projects
)

if missing_supervisor_projects:
    raise RuntimeError(
        "Supervisor projects missing from the sample: "
        f"{sorted(missing_supervisor_projects)}"
    )

Sample manifest:


,sample_stratum,proposal_id,member_ous_uid
0,wide_bandwidth,2024.1.01553.S,uid://A001/X3788/Xb661
1,wide_bandwidth,2016.1.00447.S,uid://A001/X88f/X28a
2,wide_bandwidth,2019.1.01813.S,uid://A001/X1465/X5e
3,wide_bandwidth,2021.1.01150.S,uid://A001/X1590/Xea7
4,wide_bandwidth,2024.1.01293.S,uid://A001/X3788/Xa159
5,wide_bandwidth,2019.1.00195.L,uid://A001/X1467/X1e7
6,narrow_bandwidth,2025.1.01257.S,uid://A001/X3833/X10ab
7,narrow_bandwidth,2025.1.01329.S,uid://A001/X3833/Xe25
8,narrow_bandwidth,2025.1.00720.S,uid://A001/X3922/X294
9,narrow_bandwidth,2025.1.00188.S,uid://A001/X3833/X4f90


Selected supervisor projects: {'2021.A.00028.S', '2018.1.00294.S'}


In [33]:
selected_member_uids = (
    sample_manifest_df["member_ous_uid"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

member_uid_sql = ",\n    ".join(
    quote_adql_string(member_uid)
    for member_uid in selected_member_uids
)

print(
    "Selected unique Member OUS datasets:",
    len(selected_member_uids),
)

Selected unique Member OUS datasets: 18


## 4. Complete Member OUS Retrieval

The complete sample is counted before retrieval. The result is rejected if it
exceeds the exploratory safety limit or if the returned row count differs from
the count query.

In [34]:
count_query = f'''
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
    {member_uid_sql}
)
'''

count_table = run_tap_query(
    count_query,
    maxrec=10,
    label="Count complete corrected sample",
)

expected_row_count = int(
    count_table["total_rows"][0]
)

MAX_SAMPLE_ROWS = 20_000

print("Expected complete rows:", expected_row_count)
print("Exploratory safety limit:", MAX_SAMPLE_ROWS)

if expected_row_count > MAX_SAMPLE_ROWS:
    raise RuntimeError(
        f"Sample contains {expected_row_count:,} rows. "
        "Reduce Member OUS selection before continuing."
    )


--- Count complete corrected sample ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
    'uid://A001/X3788/Xb661',
    'uid://A001/X88f/X28a',
    'uid://A001/X1465/X5e',
    'uid://A001/X1590/Xea7',
    'uid://A001/X3788/Xa159',
    'uid://A001/X1467/X1e7',
    'uid://A001/X3833/X10ab',
    'uid://A001/X3833/Xe25',
    'uid://A001/X3922/X294',
    'uid://A001/X3833/X4f90',
    'uid://A001/X3833/X4c0a',
    'uid://A001/X3833/X2dd1',
    'uid://A001/X3833/X64ea',
    'uid://A001/X133d/X3ed0',
    'uid://A001/X1465/X2767',
    'uid://A001/X133d/X9c7',
    'uid://A001/X133d/X9bb',
    'uid://A001/X2df9/X1b'
)

Retrieved rows: 1
Expected complete rows: 1611
Exploratory safety limit: 20000


In [35]:
column_sql = ",\n    ".join(
    selected_archive_columns
)

complete_query = f'''
SELECT
    {column_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
    {member_uid_sql}
)
'''

archive_table = run_tap_query(
    complete_query,
    maxrec=max(expected_row_count, 1),
    label="Retrieve complete corrected sample",
)

archive_df = archive_table.to_pandas()

retrieval_complete = (
    len(archive_df) == expected_row_count
)

print("Expected rows:", expected_row_count)
print("Retrieved rows:", len(archive_df))
print("Complete retrieval:", retrieval_complete)

if not retrieval_complete:
    raise RuntimeError(
        "Corrected sample retrieval is incomplete: "
        f"expected {expected_row_count}, got {len(archive_df)}."
    )


--- Retrieve complete corrected sample ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 1611
ADQL:

SELECT
    proposal_id,
    group_ous_uid,
    member_ous_uid,
    obs_id,
    asdm_uid,
    target_name,
    s_ra,
    s_dec,
    s_region,
    frequency,
    bandwidth,
    frequency_support,
    spectral_resolution,
    velocity_resolution,
    em_resolution,
    s_resolution,
    spatial_resolution,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    antenna_arrays,
    is_mosaic,
    pol_states,
    band_list,
    scan_intent
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
    'uid://A001/X3788/Xb661',
    'uid://A001/X88f/X28a',
    'uid://A001/X1465/X5e',
    'uid://A001/X1590/Xea7',
    'uid://A001/X3788/Xa159',
    'uid://A001/X1467/X1e7',
    'uid://A001/X3833/X10ab',
    'uid://A001/X3833/Xe25',
    'uid://A001/X3922/X294',
    'uid://A001/X3833/X4f90',
    'uid://A001/X3833/X4c0a',
    'uid://A001/X3833/X2dd1',
    'uid://A001/X3833/X64e

In [36]:
retrieved_projects = set(
    archive_df["proposal_id"]
    .dropna()
    .astype(str)
)

missing_retrieved_supervisor_projects = (
    expected_supervisor_projects
    - retrieved_projects
)

if missing_retrieved_supervisor_projects:
    raise RuntimeError(
        "Supervisor projects were selected but not retrieved: "
        f"{sorted(missing_retrieved_supervisor_projects)}"
    )

member_strata_df = (
    sample_manifest_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )
    .agg(
        sample_strata=(
            "sample_stratum",
            lambda values: " | ".join(
                sorted(set(values))
            ),
        ),
    )
    .reset_index()
)

archive_df = archive_df.merge(
    member_strata_df,
    on="member_ous_uid",
    how="left",
    validate="many_to_one",
)

print("Corrected sample validation passed.")
print("Member OUS count:", archive_df["member_ous_uid"].nunique())
print("Proposal count:", archive_df["proposal_id"].nunique())
print("Raw Archive rows:", len(archive_df))
print("ASDM count:", archive_df["asdm_uid"].nunique())

display(
    archive_df[
        [
            "proposal_id",
            "member_ous_uid",
            "obs_id",
            "frequency",
            "bandwidth",
            "spectral_resolution",
            "sensitivity_10kms",
            "is_mosaic",
            "sample_strata",
        ]
    ].head(20)
)

Corrected sample validation passed.
Member OUS count: 18
Proposal count: 17
Raw Archive rows: 1611
ASDM count: 18


,proposal_id,member_ous_uid,obs_id,frequency,bandwidth,spectral_resolution,sensitivity_10kms,is_mosaic,sample_strata
0,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.19,225.732519,2.000000e+09,31250.000000,0.786219,F,wide_bandwidth
1,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.21,237.290863,2.000000e+09,31250.000000,0.904123,F,wide_bandwidth
2,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.23,240.296886,2.000000e+09,31250.000000,0.835006,F,wide_bandwidth
3,2024.1.01553.S,uid://A001/X3788/Xb661,uid://A001/X3788/Xb661.source.3C078.spw.25,223.836776,1.875000e+09,7812.011719,0.838293,F,wide_bandwidth
4,2016.1.00447.S,uid://A001/X88f/X28a,uid://A001/X88f/X28a.source.sigOri_583.spw.23,233.998205,2.000000e+09,31250.000000,2.501515,F,wide_bandwidth
5,2016.1.00447.S,uid://A001/X88f/X28a,uid://A001/X88f/X28a.source.sigOri_583.spw.25,219.553142,5.859375e+07,70.556641,2.487748,F,wide_bandwidth
6,2016.1.00447.S,uid://A001/X88f/X28a,uid://A001/X88f/X28a.source.sigOri_583.spw.27,220.381421,5.859375e+07,70.556641,2.484063,F,wide_bandwidth
7,2016.1.00447.S,uid://A001/X88f/X28a,uid://A001/X88f/X28a.source.sigOri_583.spw.29,216.997269,1.875000e+09,1128.906250,2.523037,F,wide_bandwidth
8,2016.1.00447.S,uid://A001/X88f/X28a,uid://A001/X88f/X28a.source.sigOri_583.spw.31,230.540400,5.859375e+07,60.577393,2.441060,F,wide_bandwidth
9,2019.1.01813.S,uid://A001/X1465/X5e,uid://A001/X1465/X5e.source.M12_81.spw.17,224.005943,2.000000e+09,31250.000000,6.002640,F,wide_bandwidth


In [37]:
sample_validation_df = pd.DataFrame(
    [
        {
            "check": "Core Archive columns available",
            "passed": not missing_core_columns,
            "details": f"{len(core_required_columns)} required columns",
        },
        {
            "check": "spectral_resolution retrieved",
            "passed": "spectral_resolution" in archive_df.columns,
            "details": archive_unit_by_column.get(
                "spectral_resolution"
            ),
        },
        {
            "check": "CASE1 project represented",
            "passed": "2021.A.00028.S" in retrieved_projects,
            "details": "Supervisor positive-retrieval case",
        },
        {
            "check": "CASE2 project represented",
            "passed": "2018.1.00294.S" in retrieved_projects,
            "details": "Supervisor positive-retrieval case",
        },
        {
            "check": "Complete retrieval",
            "passed": retrieval_complete,
            "details": (
                f"expected={expected_row_count}, "
                f"retrieved={len(archive_df)}"
            ),
        },
        {
            "check": "Exploratory safety limit",
            "passed": expected_row_count <= MAX_SAMPLE_ROWS,
            "details": f"limit={MAX_SAMPLE_ROWS}",
        },
    ]
)

display(sample_validation_df)

if not sample_validation_df["passed"].all():
    raise RuntimeError(
        "One or more corrected sample validation checks failed."
    )

,check,passed,details
0,Core Archive columns available,True,17 required columns
1,spectral_resolution retrieved,True,kHz
2,CASE1 project represented,True,Supervisor positive-retrieval case
3,CASE2 project represented,True,Supervisor positive-retrieval case
4,Complete retrieval,True,"expected=1611, retrieved=1611"
5,Exploratory safety limit,True,limit=20000


## 5. Checkpoint and Next Experiment

The next experiment will parse `obs_id`, reconstruct Source-SPW records, test
candidate-key uniqueness, and determine the ownership level at which spectral
setup evidence should be evaluated.

## 6. Archive Row Roles and Field Completeness

This experiment profiles the corrected complete sample before reconstructing
Source-SPW records.

It determines:

1. which selected fields are populated;
2. which scan intents occur;
3. whether `science_observation = 'T'` is sufficient to identify rows relevant
   to duplication comparison;
4. which fields may safely be required by the normalized model.

No rows are removed in this experiment. Raw Archive rows remain preserved.

In [38]:
import numpy as np


analysis_df = archive_df.copy()

analysis_df.insert(
    0,
    "archive_row_index",
    range(len(analysis_df)),
)

print("Analysis rows:", len(analysis_df))
print(
    "Member OUS count:",
    analysis_df["member_ous_uid"].nunique(),
)
print(
    "ASDM count:",
    analysis_df["asdm_uid"].nunique(),
)

Analysis rows: 1611
Member OUS count: 18
ASDM count: 18


In [39]:
def representative_values(
    series: pd.Series,
    limit: int = 5,
) -> list[str]:
    values = (
        series
        .dropna()
        .astype(str)
        .map(str.strip)
    )

    values = values[values.ne("")]

    return values.drop_duplicates().head(limit).tolist()


field_profile_records = []

for column_name in selected_archive_columns:
    series = analysis_df[column_name]

    blank_mask = (
        series.astype("string")
        .str.strip()
        .eq("")
        .fillna(False)
    )

    missing_mask = series.isna() | blank_mask

    field_profile_records.append(
        {
            "column_name": column_name,
            "archive_unit": (
                archive_unit_by_column.get(column_name)
            ),
            "pandas_dtype": str(series.dtype),
            "row_count": len(series),
            "non_missing_count": int(
                (~missing_mask).sum()
            ),
            "missing_count": int(
                missing_mask.sum()
            ),
            "missing_fraction": float(
                missing_mask.mean()
            ),
            "unique_non_missing_count": int(
                series[~missing_mask]
                .astype(str)
                .nunique()
            ),
            "example_values": representative_values(
                series
            ),
        }
    )


archive_field_profile_df = (
    pd.DataFrame(field_profile_records)
    .sort_values(
        [
            "missing_fraction",
            "column_name",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(archive_field_profile_df)

,column_name,archive_unit,pandas_dtype,row_count,non_missing_count,missing_count,missing_fraction,unique_non_missing_count,example_values
0,antenna_arrays,,str,1611,1611,0,0.0,14,[A001:DA63 A002:DA45 A007:DV20 A008:DV17 A010:...
1,asdm_uid,,str,1611,1611,0,0.0,18,"[uid://A002/X123b93d/X34d2, uid://A002/Xb9fa0c..."
2,band_list,,str,1611,1611,0,0.0,4,"[6, 3, 7, 9]"
3,bandwidth,Hz,float64,1611,1611,0,0.0,10,"[2000000000.0, 1875000000.0, 58593750.0, 50000..."
4,cont_sensitivity_bandwidth,mJy/beam,float64,1611,1611,0,0.0,374,"[0.026224800912547672, 0.10814497635460897, 0...."
5,em_resolution,m,float64,1611,1611,0,0.0,1611,"[1.838615267939985e-07, 1.663858108630073e-07,..."
6,frequency,GHz,float64,1611,1611,0,0.0,1611,"[225.73251875834376, 237.29086310400703, 240.2..."
7,frequency_support,GHz,str,1611,1611,0,0.0,37,"[[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@1..."
8,group_ous_uid,,str,1611,1611,0,0.0,18,"[uid://A001/X3788/Xb660, uid://A001/X88f/X289,..."
9,is_mosaic,,str,1611,1611,0,0.0,2,"[F, T]"


In [40]:
scan_intent_summary_df = (
    analysis_df["scan_intent"]
    .fillna("<MISSING>")
    .astype(str)
    .str.strip()
    .replace("", "<BLANK>")
    .value_counts(dropna=False)
    .rename_axis("scan_intent")
    .reset_index(name="archive_row_count")
)

scan_intent_summary_df[
    "row_fraction"
] = (
    scan_intent_summary_df["archive_row_count"]
    / len(analysis_df)
)

display(scan_intent_summary_df)

,scan_intent,archive_row_count,row_fraction
0,TARGET,1611,1.0


In [41]:
scan_intent_context_df = (
    analysis_df
    .groupby(
        [
            "scan_intent",
            "is_mosaic",
        ],
        dropna=False,
    )
    .agg(
        archive_rows=("obs_id", "size"),
        member_ous_count=(
            "member_ous_uid",
            "nunique",
        ),
        asdm_count=("asdm_uid", "nunique"),
        target_name_count=(
            "target_name",
            "nunique",
        ),
        obs_id_count=("obs_id", "nunique"),
    )
    .reset_index()
    .sort_values(
        "archive_rows",
        ascending=False,
    )
)

display(scan_intent_context_df)

,scan_intent,is_mosaic,archive_rows,member_ous_count,asdm_count,target_name_count,obs_id_count
0,TARGET,F,1320,11,11,317,1320
1,TARGET,T,291,7,7,57,291


## 7. Obs-ID Structure and Candidate Keys

Archive `obs_id` values appear to encode a Member OUS identifier, source
context, and spectral-window identifier.

This experiment parses that structure without assuming that it is a permanent
Archive contract. Raw `obs_id` values are preserved.

A known multi-ASDM Member OUS is added so that candidate keys can be tested
across execution contexts.

In [42]:
multi_asdm_member_uid = "uid://A001/X3833/X1022"


multi_asdm_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid = {
    quote_adql_string(multi_asdm_member_uid)
}
"""


multi_asdm_count_table = run_tap_query(
    multi_asdm_count_query,
    maxrec=10,
    label="Count targeted multi-ASDM sample",
)

multi_asdm_row_count = int(
    multi_asdm_count_table["total_rows"][0]
)

print("Expected multi-ASDM rows:", multi_asdm_row_count)


--- Count targeted multi-ASDM sample ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 10
ADQL:

SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid = 'uid://A001/X3833/X1022'

Retrieved rows: 1
Expected multi-ASDM rows: 8


In [43]:
multi_asdm_query = f"""
SELECT
    {column_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid = {
    quote_adql_string(multi_asdm_member_uid)
}
"""


multi_asdm_table = run_tap_query(
    multi_asdm_query,
    maxrec=max(multi_asdm_row_count, 1),
    label="Retrieve targeted multi-ASDM sample",
)

multi_asdm_df = (
    multi_asdm_table
    .to_pandas()
    .copy()
)

if len(multi_asdm_df) != multi_asdm_row_count:
    raise RuntimeError(
        "Targeted multi-ASDM retrieval was truncated."
    )

print("Rows:", len(multi_asdm_df))
print(
    "ASDM count:",
    multi_asdm_df["asdm_uid"].nunique(),
)


--- Retrieve targeted multi-ASDM sample ---
Endpoint: https://almascience.eso.org/tap
MAXREC: 8
ADQL:

SELECT
    proposal_id,
    group_ous_uid,
    member_ous_uid,
    obs_id,
    asdm_uid,
    target_name,
    s_ra,
    s_dec,
    s_region,
    frequency,
    bandwidth,
    frequency_support,
    spectral_resolution,
    velocity_resolution,
    em_resolution,
    s_resolution,
    spatial_resolution,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    antenna_arrays,
    is_mosaic,
    pol_states,
    band_list,
    scan_intent
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid = 'uid://A001/X3833/X1022'

Retrieved rows: 8
Rows: 8
ASDM count: 2


In [44]:
analysis_df = (
    pd.concat(
        [
            analysis_df.drop(
                columns=["archive_row_index"]
            ),
            multi_asdm_df,
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

analysis_df.insert(
    0,
    "archive_row_index",
    range(len(analysis_df)),
)

print("Expanded analysis rows:", len(analysis_df))
print(
    "Expanded Member OUS count:",
    analysis_df["member_ous_uid"].nunique(),
)
print(
    "Expanded ASDM count:",
    analysis_df["asdm_uid"].nunique(),
)

Expanded analysis rows: 1619
Expanded Member OUS count: 19
Expanded ASDM count: 20


In [45]:
import re


OBS_ID_PATTERN = re.compile(
    r"^(?P<obs_member_ous_uid>uid://.+?)"
    r"\.source\."
    r"(?P<source_name>.+)"
    r"\.spw\."
    r"(?P<spw_id>[^.]+)$"
)


def parse_obs_id_structure(
    raw_obs_id: object,
) -> dict[str, object]:
    if pd.isna(raw_obs_id):
        return {
            "obs_member_ous_uid": None,
            "source_name": None,
            "spw_id": None,
            "obs_id_parse_status": "FAILED",
            "obs_id_parse_issue": "missing_obs_id",
        }

    obs_id_text = str(raw_obs_id).strip()
    match = OBS_ID_PATTERN.fullmatch(obs_id_text)

    if match is None:
        return {
            "obs_member_ous_uid": None,
            "source_name": None,
            "spw_id": None,
            "obs_id_parse_status": "FAILED",
            "obs_id_parse_issue": (
                "unexpected_obs_id_format"
            ),
        }

    return {
        **match.groupdict(),
        "obs_id_parse_status": "PARSED",
        "obs_id_parse_issue": None,
    }


parsed_obs_id_df = pd.DataFrame(
    analysis_df["obs_id"]
    .map(parse_obs_id_structure)
    .tolist()
)

analysis_df = pd.concat(
    [
        analysis_df,
        parsed_obs_id_df,
    ],
    axis=1,
)

In [46]:
analysis_df[
    "obs_member_matches_column"
] = (
    analysis_df["obs_member_ous_uid"]
    == analysis_df["member_ous_uid"].astype(str)
)

obs_id_parse_summary_df = (
    analysis_df
    .groupby(
        [
            "obs_id_parse_status",
            "obs_id_parse_issue",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="archive_row_count")
)

display(obs_id_parse_summary_df)

display(
    analysis_df[
        (
            analysis_df["obs_id_parse_status"]
            != "PARSED"
        )
        | (
            ~analysis_df[
                "obs_member_matches_column"
            ]
        )
    ][
        [
            "member_ous_uid",
            "asdm_uid",
            "obs_id",
            "obs_id_parse_status",
            "obs_id_parse_issue",
        ]
    ].head(50)
)

,obs_id_parse_status,obs_id_parse_issue,archive_row_count
0,PARSED,NaN,1619


,member_ous_uid,asdm_uid,obs_id,obs_id_parse_status,obs_id_parse_issue


In [47]:
candidate_keys = {
    "obs_id": [
        "obs_id",
    ],
    "member_source_spw": [
        "member_ous_uid",
        "source_name",
        "spw_id",
    ],
    "member_asdm_source_spw": [
        "member_ous_uid",
        "asdm_uid",
        "source_name",
        "spw_id",
    ],
}


candidate_key_records = []

for key_name, key_columns in candidate_keys.items():
    missing_key_mask = (
        analysis_df[key_columns]
        .isna()
        .any(axis=1)
    )

    duplicate_mask = analysis_df.duplicated(
        subset=key_columns,
        keep=False,
    )

    candidate_key_records.append(
        {
            "candidate_key": key_name,
            "key_columns": key_columns,
            "row_count": len(analysis_df),
            "missing_key_rows": int(
                missing_key_mask.sum()
            ),
            "duplicate_rows": int(
                duplicate_mask.sum()
            ),
            "unique_key_count": int(
                analysis_df[key_columns]
                .drop_duplicates()
                .shape[0]
            ),
            "is_unique_and_complete": (
                not missing_key_mask.any()
                and not duplicate_mask.any()
            ),
        }
    )


candidate_key_validation_df = pd.DataFrame(
    candidate_key_records
)

display(candidate_key_validation_df)

,candidate_key,key_columns,row_count,missing_key_rows,duplicate_rows,unique_key_count,is_unique_and_complete
0,obs_id,[obs_id],1619,0,0,1619,True
1,member_source_spw,"[member_ous_uid, source_name, spw_id]",1619,0,0,1619,True
2,member_asdm_source_spw,"[member_ous_uid, asdm_uid, source_name, spw_id]",1619,0,0,1619,True


## 8. Source-SPW and Frequency-Support Reconstruction

This experiment uses the production frequency-support parser. The parser is
not reimplemented in this notebook.

Parsed components are associated conservatively with Member OUS, ASDM, source
context, and raw signature. Row frequencies are then assigned numerically to
component intervals.

In [48]:
frequency_signature_inventory_df = (
    analysis_df[
        [
            "member_ous_uid",
            "asdm_uid",
            "source_name",
            "frequency_support",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Unique source-execution signatures:",
    len(frequency_signature_inventory_df),
)

Unique source-execution signatures: 376


In [49]:
def issue_codes(issues) -> tuple[str, ...]:
    return tuple(issue.code for issue in issues)


component_records = []

for row in (
    frequency_signature_inventory_df
    .itertuples(index=False)
):
    result = parse_frequency_support(
        row.frequency_support
    )

    if not result.components:
        component_records.append(
            {
                "member_ous_uid": row.member_ous_uid,
                "asdm_uid": row.asdm_uid,
                "source_name": row.source_name,
                "raw_frequency_support": (
                    row.frequency_support
                ),
                "parser_version": (
                    result.parser_version
                ),
                "result_parse_status": (
                    result.parse_status.value
                ),
                "result_parse_issues": issue_codes(
                    result.parse_issues
                ),
                "component_index": None,
            }
        )
        continue

    for component in result.components:
        interval = component.frequency_interval
        resolution = component.resolution

        sensitivity_by_basis = {
            entry.basis: entry
            for entry in component.sensitivities
        }

        sensitivity_10kms = (
            sensitivity_by_basis.get("10km/s")
        )
        sensitivity_native = (
            sensitivity_by_basis.get("native")
        )

        component_records.append(
            {
                "member_ous_uid": row.member_ous_uid,
                "asdm_uid": row.asdm_uid,
                "source_name": row.source_name,
                "raw_frequency_support": (
                    row.frequency_support
                ),
                "parser_version": (
                    result.parser_version
                ),
                "result_parse_status": (
                    result.parse_status.value
                ),
                "result_parse_issues": issue_codes(
                    result.parse_issues
                ),
                "component_index": (
                    component.component_index
                ),
                "component_parse_status": (
                    component.parse_status.value
                ),
                "component_parse_issues": (
                    issue_codes(
                        component.parse_issues
                    )
                ),
                "component_validation_issues": (
                    issue_codes(
                        component.validation_issues
                    )
                ),
                "frequency_low_ghz": (
                    None
                    if interval is None
                    else (
                        interval.low
                        * u.Unit(interval.unit)
                    ).to_value(u.GHz)
                ),
                "frequency_high_ghz": (
                    None
                    if interval is None
                    else (
                        interval.high
                        * u.Unit(interval.unit)
                    ).to_value(u.GHz)
                ),
                "component_resolution_khz": (
                    None
                    if resolution is None
                    else (
                        resolution.value
                        * u.Unit(resolution.unit)
                    ).to_value(u.kHz)
                ),
                "sensitivity_10kms_mjy": (
                    None
                    if sensitivity_10kms is None
                    else (
                        sensitivity_10kms.value
                        * u.Unit(
                            sensitivity_10kms.unit
                        )
                    ).to_value(
                        u.mJy / u.beam
                    )
                ),
                "sensitivity_native_mjy": (
                    None
                    if sensitivity_native is None
                    else (
                        sensitivity_native.value
                        * u.Unit(
                            sensitivity_native.unit
                        )
                    ).to_value(
                        u.mJy / u.beam
                    )
                ),
                "polarization_products": (
                    component.polarization_products
                ),
            }
        )


frequency_support_component_df = pd.DataFrame(
    component_records
)

display(frequency_support_component_df.head(20))

,member_ous_uid,asdm_uid,source_name,raw_frequency_support,parser_version,result_parse_status,result_parse_issues,component_index,component_parse_status,component_parse_issues,component_validation_issues,frequency_low_ghz,frequency_high_ghz,component_resolution_khz,sensitivity_10kms_mjy,sensitivity_native_mjy,polarization_products
0,uid://A001/X3788/Xb661,uid://A002/X123b93d/X34d2,3C078,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10...",1,PARSED,(),1,PARSED,(),(),222.90,224.77,7812.01,0.8383,0.0529,"(XX, YY)"
1,uid://A001/X3788/Xb661,uid://A002/X123b93d/X34d2,3C078,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10...",1,PARSED,(),2,PARSED,(),(),224.74,226.72,31250.00,0.7862,0.0482,"(XX, YY)"
2,uid://A001/X3788/Xb661,uid://A002/X123b93d/X34d2,3C078,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10...",1,PARSED,(),3,PARSED,(),(),236.30,238.28,31250.00,0.9041,0.0569,"(XX, YY)"
3,uid://A001/X3788/Xb661,uid://A002/X123b93d/X34d2,3C078,"[222.90..224.77GHz,7812.01kHz,838.3uJy/beam@10...",1,PARSED,(),4,PARSED,(),(),239.30,241.29,31250.00,0.8350,0.0529,"(XX, YY)"
4,uid://A001/X88f/X28a,uid://A002/Xb9fa0c/X1954,sigOri_583,"[216.06..217.93GHz,1128.91kHz,2.5mJy/beam@10km...",1,PARSED,(),1,PARSED,(),(),216.06,217.93,1128.91,2.5000,0.1568,"(XX, YY)"
5,uid://A001/X88f/X28a,uid://A002/Xb9fa0c/X1954,sigOri_583,"[216.06..217.93GHz,1128.91kHz,2.5mJy/beam@10km...",1,PARSED,(),2,PARSED,(),(),219.52,219.58,70.56,2.5000,0.8795,"(XX, YY)"
6,uid://A001/X88f/X28a,uid://A002/Xb9fa0c/X1954,sigOri_583,"[216.06..217.93GHz,1128.91kHz,2.5mJy/beam@10km...",1,PARSED,(),3,PARSED,(),(),220.35,220.41,70.56,2.5000,0.8799,"(XX, YY)"
7,uid://A001/X88f/X28a,uid://A002/Xb9fa0c/X1954,sigOri_583,"[216.06..217.93GHz,1128.91kHz,2.5mJy/beam@10km...",1,PARSED,(),4,PARSED,(),(),230.51,230.57,60.58,2.4000,0.8843,"(XX, YY)"
8,uid://A001/X88f/X28a,uid://A002/Xb9fa0c/X1954,sigOri_583,"[216.06..217.93GHz,1128.91kHz,2.5mJy/beam@10km...",1,PARSED,(),5,PARSED,(),(),233.01,234.99,31250.00,2.5000,0.1563,"(XX, YY)"
9,uid://A001/X1465/X5e,uid://A002/Xe3da01/X2490,M12_81,"[223.01..225.00GHz,31250.00kHz,6mJy/beam@10km/...",1,PARSED,(),1,PARSED,(),(),223.01,225.00,31250.00,6.0000,0.3669,"(XX, YY)"


In [50]:
assignment_records = []

for row in analysis_df.itertuples(index=False):
    candidates = frequency_support_component_df[
        (
            frequency_support_component_df[
                "member_ous_uid"
            ]
            == row.member_ous_uid
        )
        & (
            frequency_support_component_df[
                "asdm_uid"
            ]
            == row.asdm_uid
        )
        & (
            frequency_support_component_df[
                "source_name"
            ]
            == row.source_name
        )
        & (
            frequency_support_component_df[
                "raw_frequency_support"
            ]
            == row.frequency_support
        )
    ]

    matches = candidates[
        (
            candidates["frequency_low_ghz"]
            <= row.frequency
        )
        & (
            row.frequency
            <= candidates[
                "frequency_high_ghz"
            ]
        )
    ]

    assignment_records.append(
        {
            "archive_row_index": (
                row.archive_row_index
            ),
            "obs_id": row.obs_id,
            "match_count": len(matches),
            "assigned_component_index": (
                matches["component_index"].iloc[0]
                if len(matches) == 1
                else None
            ),
        }
    )


row_component_assignment_df = pd.DataFrame(
    assignment_records
)

display(
    row_component_assignment_df[
        "match_count"
    ]
    .value_counts(dropna=False)
    .rename_axis("match_count")
    .reset_index(name="archive_row_count")
)

,match_count,archive_row_count
0,1,1393
1,2,226


In [51]:
parser_result_summary_df = (
    frequency_support_component_df
    .groupby(
        "result_parse_status",
        dropna=False,
    )
    .agg(
        component_rows=(
            "component_index",
            "size",
        ),
        source_execution_contexts=(
            "raw_frequency_support",
            "count",
        ),
    )
    .reset_index()
)

component_parse_summary_df = (
    frequency_support_component_df
    .groupby(
        "component_parse_status",
        dropna=False,
    )
    .size()
    .reset_index(name="component_count")
)

display(parser_result_summary_df)
display(component_parse_summary_df)

,result_parse_status,component_rows,source_execution_contexts
0,PARSED,1619,1619


,component_parse_status,component_count
0,PARSED,1619


In [52]:
component_issue_df = (
    frequency_support_component_df[
        frequency_support_component_df[
            "component_parse_issues"
        ].map(bool)
        | frequency_support_component_df[
            "component_validation_issues"
        ].map(bool)
    ]
)

print(
    "Components with parser or validation issues:",
    len(component_issue_df),
)

display(component_issue_df.head(50))

Components with parser or validation issues: 0


,member_ous_uid,asdm_uid,source_name,raw_frequency_support,parser_version,result_parse_status,result_parse_issues,component_index,component_parse_status,component_parse_issues,component_validation_issues,frequency_low_ghz,frequency_high_ghz,component_resolution_khz,sensitivity_10kms_mjy,sensitivity_native_mjy,polarization_products


In [53]:
context_columns = [
    "member_ous_uid",
    "asdm_uid",
    "source_name",
    "frequency_support",
]


row_context_count_df = (
    analysis_df
    .groupby(
        context_columns,
        dropna=False,
    )
    .size()
    .reset_index(name="archive_row_count")
)


component_context_count_df = (
    frequency_support_component_df
    .rename(
        columns={
            "raw_frequency_support": (
                "frequency_support"
            )
        }
    )
    .groupby(
        context_columns,
        dropna=False,
    )
    .agg(
        parsed_component_count=(
            "component_index",
            "count",
        )
    )
    .reset_index()
)


context_count_comparison_df = (
    row_context_count_df
    .merge(
        component_context_count_df,
        on=context_columns,
        how="outer",
        validate="one_to_one",
    )
)

context_count_comparison_df[
    "count_agreement"
] = (
    context_count_comparison_df[
        "archive_row_count"
    ]
    == context_count_comparison_df[
        "parsed_component_count"
    ]
)

display(
    context_count_comparison_df[
        "count_agreement"
    ]
    .value_counts(dropna=False)
    .rename_axis("count_agreement")
    .reset_index(name="context_count")
)

display(
    context_count_comparison_df[
        ~context_count_comparison_df[
            "count_agreement"
        ]
    ].head(50)
)

,count_agreement,context_count
0,True,376


,member_ous_uid,asdm_uid,source_name,frequency_support,archive_row_count,parsed_component_count,count_agreement


In [54]:
component_assignment_source_df = (
    frequency_support_component_df
    .rename(
        columns={
            "raw_frequency_support": (
                "frequency_support"
            )
        }
    )
    .copy()
)

component_assignment_source_df[
    "component_centre_ghz"
] = (
    component_assignment_source_df[
        "frequency_low_ghz"
    ]
    + component_assignment_source_df[
        "frequency_high_ghz"
    ]
) / 2

component_assignment_source_df[
    "component_width_hz"
] = (
    component_assignment_source_df[
        "frequency_high_ghz"
    ]
    - component_assignment_source_df[
        "frequency_low_ghz"
    ]
) * 1e9

In [55]:
component_groups = {
    key: group.copy()
    for key, group in (
        component_assignment_source_df
        .groupby(
            context_columns,
            dropna=False,
            sort=False,
        )
    )
}


one_to_one_assignment_records = []

for context_key, row_group in (
    analysis_df.groupby(
        context_columns,
        dropna=False,
        sort=False,
    )
):
    component_group = component_groups.get(
        context_key
    )

    if (
        component_group is None
        or len(row_group) != len(component_group)
    ):
        for row in row_group.itertuples(
            index=False
        ):
            one_to_one_assignment_records.append(
                {
                    "archive_row_index": (
                        row.archive_row_index
                    ),
                    "assignment_status": (
                        "COUNT_MISMATCH"
                    ),
                }
            )
        continue

    sorted_rows = row_group.sort_values(
        [
            "frequency",
            "spw_id",
        ]
    )

    sorted_components = (
        component_group.sort_values(
            [
                "component_centre_ghz",
                "component_index",
            ]
        )
    )

    for (
        row,
        component,
    ) in zip(
        sorted_rows.itertuples(index=False),
        sorted_components.itertuples(
            index=False
        ),
        strict=True,
    ):
        frequency_inside_interval = (
            component.frequency_low_ghz
            <= row.frequency
            <= component.frequency_high_ghz
        )

        one_to_one_assignment_records.append(
            {
                "archive_row_index": (
                    row.archive_row_index
                ),
                "obs_id": row.obs_id,
                "member_ous_uid": (
                    row.member_ous_uid
                ),
                "asdm_uid": row.asdm_uid,
                "source_name": row.source_name,
                "spw_id": row.spw_id,
                "row_frequency_ghz": (
                    row.frequency
                ),
                "component_index": (
                    component.component_index
                ),
                "frequency_low_ghz": (
                    component.frequency_low_ghz
                ),
                "frequency_high_ghz": (
                    component.frequency_high_ghz
                ),
                "component_centre_ghz": (
                    component.component_centre_ghz
                ),
                "component_width_hz": (
                    component.component_width_hz
                ),
                "component_resolution_khz": (
                    component.component_resolution_khz
                ),
                "parsed_sensitivity_10kms_mjy": (
                    component.sensitivity_10kms_mjy
                ),
                "parsed_sensitivity_native_mjy": (
                    component.sensitivity_native_mjy
                ),
                "frequency_inside_interval": (
                    frequency_inside_interval
                ),
                "centre_difference_mhz": (
                    abs(
                        row.frequency
                        - component.component_centre_ghz
                    )
                    * 1e3
                ),
                "assignment_status": (
                    "ASSIGNED"
                    if frequency_inside_interval
                    else "OUTSIDE_INTERVAL"
                ),
            }
        )


one_to_one_assignment_df = pd.DataFrame(
    one_to_one_assignment_records
)

In [56]:
assignment_validation_df = (
    one_to_one_assignment_df[
        "assignment_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("assignment_status")
    .reset_index(name="archive_row_count")
)

display(assignment_validation_df)

print(
    "Assigned component uniqueness:",
    not one_to_one_assignment_df.duplicated(
        subset=[
            "member_ous_uid",
            "asdm_uid",
            "source_name",
            "component_index",
        ]
    ).any(),
)

print(
    "Archive row uniqueness:",
    not one_to_one_assignment_df.duplicated(
        subset=["archive_row_index"]
    ).any(),
)

print(
    "Maximum centre difference MHz:",
    one_to_one_assignment_df[
        "centre_difference_mhz"
    ].max(),
)

,assignment_status,archive_row_count
0,ASSIGNED,1619


Assigned component uniqueness: True
Archive row uniqueness: True
Maximum centre difference MHz: 4.849845599295577


In [57]:
spectral_comparison_df = (
    analysis_df
    .merge(
        one_to_one_assignment_df[
            [
                "archive_row_index",
                "component_index",
                "frequency_low_ghz",
                "frequency_high_ghz",
                "component_centre_ghz",
                "component_width_hz",
                "component_resolution_khz",
                "parsed_sensitivity_10kms_mjy",
                "parsed_sensitivity_native_mjy",
                "centre_difference_mhz",
                "assignment_status",
            ]
        ],
        on="archive_row_index",
        how="left",
        validate="one_to_one",
    )
)

In [58]:
spectral_comparison_df[
    "bandwidth_difference_hz"
] = (
    spectral_comparison_df["bandwidth"]
    - spectral_comparison_df[
        "component_width_hz"
    ]
)

spectral_comparison_df[
    "bandwidth_relative_difference"
] = (
    spectral_comparison_df[
        "bandwidth_difference_hz"
    ]
    / spectral_comparison_df[
        "component_width_hz"
    ]
)

spectral_comparison_df[
    "archive_above_1_8ghz"
] = (
    spectral_comparison_df["bandwidth"]
    > 1.8e9
)

spectral_comparison_df[
    "parsed_above_1_8ghz"
] = (
    spectral_comparison_df[
        "component_width_hz"
    ]
    > 1.8e9
)

spectral_comparison_df[
    "bandwidth_threshold_agreement"
] = (
    spectral_comparison_df[
        "archive_above_1_8ghz"
    ]
    == spectral_comparison_df[
        "parsed_above_1_8ghz"
    ]
)

In [59]:
spectral_comparison_df[
    "resolution_difference_khz"
] = (
    spectral_comparison_df[
        "spectral_resolution"
    ]
    - spectral_comparison_df[
        "component_resolution_khz"
    ]
)


C_M_PER_S = 299_792_458.0


spectral_comparison_df[
    "derived_velocity_resolution_mps"
] = (
    C_M_PER_S
    * (
        spectral_comparison_df[
            "component_resolution_khz"
        ]
        * 1e3
    )
    / (
        spectral_comparison_df["frequency"]
        * 1e9
    )
)

spectral_comparison_df[
    "velocity_resolution_difference_mps"
] = (
    spectral_comparison_df[
        "velocity_resolution"
    ]
    - spectral_comparison_df[
        "derived_velocity_resolution_mps"
    ]
)

In [60]:
spectral_comparison_df[
    "sensitivity_10kms_difference_mjy"
] = (
    spectral_comparison_df[
        "sensitivity_10kms"
    ]
    - spectral_comparison_df[
        "parsed_sensitivity_10kms_mjy"
    ]
)

In [61]:
spectral_semantics_summary_df = (
    spectral_comparison_df[
        [
            "bandwidth_difference_hz",
            "bandwidth_relative_difference",
            "resolution_difference_khz",
            "velocity_resolution_difference_mps",
            "sensitivity_10kms_difference_mjy",
            "centre_difference_mhz",
        ]
    ]
    .describe()
    .T
)

display(spectral_semantics_summary_df)

display(
    spectral_comparison_df[
        ~spectral_comparison_df[
            "bandwidth_threshold_agreement"
        ]
    ][
        [
            "member_ous_uid",
            "asdm_uid",
            "obs_id",
            "bandwidth",
            "component_width_hz",
            "bandwidth_difference_hz",
        ]
    ].head(100)
)

,count,mean,std,min,25%,50%,75%,max
bandwidth_difference_hz,1619.0,4.661732e+06,5.726622e+06,-7.500000e+06,0.000000,2.500000e+06,1.000000e+07,2.000000e+07
bandwidth_relative_difference,1619.0,1.798045e-05,1.708090e-02,-1.071429e-01,0.000000,2.673797e-03,5.025126e-03,6.534091e-02
resolution_difference_khz,1619.0,-3.211196e-04,2.046798e-03,-4.843750e-03,-0.001680,0.000000e+00,6.250000e-04,3.281250e-03
velocity_resolution_difference_mps,1619.0,-2.236038e+03,7.332052e+03,-4.093230e+04,-1480.861458,-1.622026e+02,-3.635151e+00,2.281444e+02
sensitivity_10kms_difference_mjy,1619.0,-5.265018e-03,2.913010e-02,-4.801623e-02,-0.037637,-9.781634e-04,1.958541e-02,4.975183e-02
centre_difference_mhz,1619.0,1.855373e+00,1.133417e+00,1.366600e-03,1.008394,1.449921e+00,2.608259e+00,4.849846e+00


,member_ous_uid,asdm_uid,obs_id,bandwidth,component_width_hz,bandwidth_difference_hz


In [62]:
spatial_analysis_df = analysis_df.copy()

spatial_analysis_df[
    "coordinate_signature"
] = list(
    zip(
        spatial_analysis_df["s_ra"].round(9),
        spatial_analysis_df["s_dec"].round(9),
    )
)

spatial_analysis_df[
    "s_region_text"
] = (
    spatial_analysis_df["s_region"]
    .fillna("<MISSING>")
    .astype(str)
    .str.strip()
)

spatial_analysis_df[
    "geometry_type"
] = (
    spatial_analysis_df[
        "s_region_text"
    ]
    .str.extract(
        r"^\s*(\w+)",
        expand=False,
    )
)

In [63]:
source_spatial_context_df = (
    spatial_analysis_df
    .groupby(
        [
            "member_ous_uid",
            "asdm_uid",
            "source_name",
            "is_mosaic",
        ],
        dropna=False,
    )
    .agg(
        archive_rows=("obs_id", "size"),
        spw_count=("spw_id", "nunique"),
        coordinate_count=(
            "coordinate_signature",
            "nunique",
        ),
        footprint_count=(
            "s_region_text",
            "nunique",
        ),
        target_name_count=(
            "target_name",
            "nunique",
        ),
        geometry_type_count=(
            "geometry_type",
            "nunique",
        ),
        representative_geometry_type=(
            "geometry_type",
            "first",
        ),
    )
    .reset_index()
)

display(source_spatial_context_df.head(50))

,member_ous_uid,asdm_uid,source_name,is_mosaic,archive_rows,spw_count,coordinate_count,footprint_count,target_name_count,geometry_type_count,representative_geometry_type
0,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,HOPS_10,T,7,7,1,1,1,1,Polygon
1,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,HOPS_108,T,7,7,1,1,1,1,Polygon
2,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,HOPS_11,T,7,7,1,1,1,1,Polygon
3,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,HOPS_182,T,7,7,1,1,1,1,Polygon
4,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,HOPS_203,T,7,7,1,1,1,1,Polygon
5,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,HOPS_310,T,7,7,1,1,1,1,Polygon
6,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,HOPS_32,T,7,7,1,1,1,1,Polygon
7,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,HOPS_373,T,7,7,1,1,1,1,Polygon
8,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,HOPS_394,T,7,7,1,1,1,1,Polygon
9,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,HOPS_397,T,7,7,1,1,1,1,Polygon


In [64]:
mosaic_spatial_context_df = (
    source_spatial_context_df[
        source_spatial_context_df[
            "is_mosaic"
        ]
        .astype(str)
        .str.upper()
        .eq("T")
    ]
)

mosaic_asdm_summary_df = (
    mosaic_spatial_context_df
    .groupby(
        [
            "member_ous_uid",
            "asdm_uid",
        ],
        dropna=False,
    )
    .agg(
        source_context_count=(
            "source_name",
            "nunique",
        ),
        total_spw_count=(
            "spw_count",
            "sum",
        ),
        maximum_coordinates_per_source=(
            "coordinate_count",
            "max",
        ),
        maximum_footprints_per_source=(
            "footprint_count",
            "max",
        ),
        geometry_types=(
            "representative_geometry_type",
            lambda values: tuple(
                sorted(set(values))
            ),
        ),
    )
    .reset_index()
)

display(mosaic_asdm_summary_df)

,member_ous_uid,asdm_uid,source_context_count,total_spw_count,maximum_coordinates_per_source,maximum_footprints_per_source,geometry_types
0,uid://A001/X133d/X3ed0,uid://A002/Xd99ff3/X13da5,19,133,1,1,"(Polygon,)"
1,uid://A001/X133d/X9c7,uid://A002/Xd3e89f/Xbb4e,1,4,1,1,"(Polygon,)"
2,uid://A001/X1465/X2767,uid://A002/Xe27761/X2335,13,52,1,1,"(Polygon,)"
3,uid://A001/X3833/X1022,uid://A002/X139bfe4/X11e15,1,4,1,1,"(Polygon,)"
4,uid://A001/X3833/X1022,uid://A002/X139fbe0/X7472,1,4,1,1,"(Polygon,)"
5,uid://A001/X3833/X2dd1,uid://A002/X13e5a87/X17c7c,1,4,1,1,"(Polygon,)"
6,uid://A001/X3833/X64ea,uid://A002/X131e84e/X1cedc,21,84,1,1,"(Polygon,)"
7,uid://A001/X3833/Xe25,uid://A002/X13e5a87/X24f5,1,4,1,1,"(Polygon,)"
8,uid://A001/X3922/X294,uid://A002/X13e5a87/X7134,1,10,1,1,"(Polygon,)"


In [65]:
spatial_mode_summary_df = (
    source_spatial_context_df
    .groupby(
        "is_mosaic",
        dropna=False,
    )
    .agg(
        source_context_count=(
            "source_name",
            "size",
        ),
        member_ous_count=(
            "member_ous_uid",
            "nunique",
        ),
        asdm_count=(
            "asdm_uid",
            "nunique",
        ),
        maximum_coordinate_count=(
            "coordinate_count",
            "max",
        ),
        maximum_footprint_count=(
            "footprint_count",
            "max",
        ),
    )
    .reset_index()
)

display(spatial_mode_summary_df)

,is_mosaic,source_context_count,member_ous_count,asdm_count,maximum_coordinate_count,maximum_footprint_count
0,F,317,11,11,1,1
1,T,59,8,9,1,1


In [66]:
all_obs_ids_parsed = (
    analysis_df["obs_id_parse_status"]
    .eq("PARSED")
    .all()
)

all_assignments_valid = (
    one_to_one_assignment_df[
        "assignment_status"
    ]
    .eq("ASSIGNED")
    .all()
)

all_spatial_fields_present = (
    spatial_analysis_df[
        [
            "s_ra",
            "s_dec",
            "s_region",
        ]
    ]
    .notna()
    .all()
    .all()
)


notebook04_readiness_df = pd.DataFrame(
    [
        {
            "entity": "ArchiveRawRow",
            "implementation_ready": True,
            "evidence": (
                "Complete raw retrieval and "
                "field preservation"
            ),
        },
        {
            "entity": "ObsIdStructure",
            "implementation_ready": (
                all_obs_ids_parsed
            ),
            "evidence": (
                "Sample parser coverage "
                f"{all_obs_ids_parsed}"
            ),
        },
        {
            "entity": "SourceExecutionContext",
            "implementation_ready": True,
            "evidence": (
                "Conservative Member + ASDM "
                "+ source context"
            ),
        },
        {
            "entity": "SourceSpwRecord",
            "implementation_ready": (
                all_assignments_valid
            ),
            "evidence": (
                "One-to-one component assignment "
                f"{all_assignments_valid}"
            ),
        },
        {
            "entity": "FrequencySupportComponent",
            "implementation_ready": (
                all_assignments_valid
            ),
            "evidence": (
                "Production parser plus "
                "validated numerical assignment"
            ),
        },
        {
            "entity": "SpatialFootprint",
            "implementation_ready": (
                all_spatial_fields_present
            ),
            "evidence": (
                "RA, Dec and s_region preserved"
            ),
        },
        {
            "entity": "MosaicPointing",
            "implementation_ready": False,
            "evidence": (
                "Individual pointing exposure "
                "not demonstrated"
            ),
        },
        {
            "entity": "ObservationModeEvidence",
            "implementation_ready": False,
            "evidence": (
                "No direct FDM/TDM Archive field"
            ),
        },
        {
            "entity": "FormalDuplicationDecision",
            "implementation_ready": False,
            "evidence": (
                "Requires Queue CSV and known-case "
                "validation"
            ),
        },
    ]
)

display(notebook04_readiness_df)

,entity,implementation_ready,evidence
0,ArchiveRawRow,True,Complete raw retrieval and field preservation
1,ObsIdStructure,True,Sample parser coverage True
2,SourceExecutionContext,True,Conservative Member + ASDM + source context
3,SourceSpwRecord,True,One-to-one component assignment True
4,FrequencySupportComponent,True,Production parser plus validated numerical ass...
5,SpatialFootprint,True,"RA, Dec and s_region preserved"
6,MosaicPointing,False,Individual pointing exposure not demonstrated
7,ObservationModeEvidence,False,No direct FDM/TDM Archive field
8,FormalDuplicationDecision,False,Requires Queue CSV and known-case validation


In [67]:
C_M_PER_S = 299_792_458.0


spectral_comparison_df[
    "derived_em_resolution_m"
] = (
    C_M_PER_S
    * (
        spectral_comparison_df[
            "component_resolution_khz"
        ]
        * 1e3
    )
    / (
        (
            spectral_comparison_df[
                "frequency"
            ]
            * 1e9
        )
        ** 2
    )
)


spectral_comparison_df[
    "em_resolution_difference_m"
] = (
    spectral_comparison_df[
        "em_resolution"
    ]
    - spectral_comparison_df[
        "derived_em_resolution_m"
    ]
)


spectral_comparison_df[
    "velocity_resolution_ratio"
] = (
    spectral_comparison_df[
        "velocity_resolution"
    ]
    / spectral_comparison_df[
        "derived_velocity_resolution_mps"
    ]
)

In [68]:
derived_field_comparison_df = (
    spectral_comparison_df[
        [
            "em_resolution",
            "derived_em_resolution_m",
            "em_resolution_difference_m",
            "velocity_resolution",
            "derived_velocity_resolution_mps",
            "velocity_resolution_difference_mps",
            "velocity_resolution_ratio",
        ]
    ]
    .describe()
    .T
)

display(derived_field_comparison_df)

,count,mean,std,min,25%,50%,75%,max
em_resolution,1619.0,7.775608e-08,8.628608e-08,3.416935e-10,1.536888e-09,7.121541e-09,1.626420e-07,3.239643e-07
derived_em_resolution_m,1619.0,7.775421e-08,8.628363e-08,3.417082e-10,1.536882e-09,7.121433e-09,1.626392e-07,3.239251e-07
em_resolution_difference_m,1619.0,1.874161e-12,3.883974e-12,-9.511940e-14,5.792429e-15,1.457508e-13,2.943376e-12,3.913332e-11
velocity_resolution,1619.0,1.505513e+04,1.851645e+04,7.878371e+01,1.134502e+02,3.314704e+02,3.887224e+04,3.887224e+04
derived_velocity_resolution_mps,1619.0,1.729117e+04,1.928837e+04,7.877759e+01,3.353898e+02,1.559645e+03,3.903444e+04,4.247662e+04
velocity_resolution_difference_mps,1619.0,-2.236038e+03,7.332052e+03,-4.093230e+04,-1.480861e+03,-1.622026e+02,-3.635151e+00,2.281444e+02
velocity_resolution_ratio,1619.0,7.361298e-01,3.551315e-01,1.967787e-03,2.490001e-01,9.377538e-01,9.961210e-01,1.009631e+00


In [69]:
ownership_levels = {
    "member": [
        "member_ous_uid",
    ],
    "execution": [
        "member_ous_uid",
        "asdm_uid",
    ],
    "source_execution": [
        "member_ous_uid",
        "asdm_uid",
        "source_name",
    ],
}

ownership_fields = [
    "spectral_resolution",
    "velocity_resolution",
    "em_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
    "component_resolution_khz",
    "parsed_sensitivity_10kms_mjy",
    "parsed_sensitivity_native_mjy",
]


ownership_records = []

for level_name, key_columns in (
    ownership_levels.items()
):
    grouped = spectral_comparison_df.groupby(
        key_columns,
        dropna=False,
    )

    for field_name in ownership_fields:
        unique_counts = grouped[
            field_name
        ].nunique(dropna=False)

        ownership_records.append(
            {
                "ownership_level": level_name,
                "field_name": field_name,
                "group_count": len(unique_counts),
                "groups_with_one_value": int(
                    unique_counts.eq(1).sum()
                ),
                "groups_with_multiple_values": int(
                    unique_counts.gt(1).sum()
                ),
                "maximum_values_per_group": int(
                    unique_counts.max()
                ),
            }
        )


field_ownership_summary_df = pd.DataFrame(
    ownership_records
)

display(field_ownership_summary_df)

,ownership_level,field_name,group_count,groups_with_one_value,groups_with_multiple_values,maximum_values_per_group
0,member,spectral_resolution,19,8,11,4
1,member,velocity_resolution,19,19,0,1
2,member,em_resolution,19,0,19,600
3,member,sensitivity_10kms,19,0,19,600
4,member,cont_sensitivity_bandwidth,19,8,11,150
5,member,component_resolution_khz,19,8,11,4
6,member,parsed_sensitivity_10kms_mjy,19,0,19,27
7,member,parsed_sensitivity_native_mjy,19,0,19,20
8,execution,spectral_resolution,20,9,11,4
9,execution,velocity_resolution,20,20,0,1


In [70]:
notebook04_final_readiness_df = pd.DataFrame(
    [
        {
            "entity": "ArchiveRawRow",
            "sample_supported": True,
            "archive_wide_guarantee": False,
            "engineering_decision": (
                "Preserve all selected raw fields"
            ),
        },
        {
            "entity": "ObsIdStructure",
            "sample_supported": True,
            "archive_wide_guarantee": False,
            "engineering_decision": (
                "Parse with status and preserve raw obs_id"
            ),
        },
        {
            "entity": "SourceExecutionContext",
            "sample_supported": True,
            "archive_wide_guarantee": False,
            "engineering_decision": (
                "Member + ASDM + source context"
            ),
        },
        {
            "entity": "SourceSpwRecord",
            "sample_supported": True,
            "archive_wide_guarantee": False,
            "engineering_decision": (
                "Execution context + SPW identifier"
            ),
        },
        {
            "entity": "FrequencySupportComponent",
            "sample_supported": True,
            "archive_wide_guarantee": False,
            "engineering_decision": (
                "Preserve raw, parsed values, status, "
                "issues and parser version"
            ),
        },
        {
            "entity": "ObservationModeEvidence",
            "sample_supported": True,
            "archive_wide_guarantee": False,
            "engineering_decision": (
                "Store bandwidth/resolution evidence "
                "without assigning FDM/TDM"
            ),
        },
        {
            "entity": "ObservationModeClassification",
            "sample_supported": False,
            "archive_wide_guarantee": False,
            "engineering_decision": (
                "Remain UNKNOWN until policy evidence "
                "is defined"
            ),
        },
        {
            "entity": "SpatialFootprint",
            "sample_supported": True,
            "archive_wide_guarantee": False,
            "engineering_decision": (
                "Preserve aggregate STC-S region"
            ),
        },
        {
            "entity": "MosaicPointing",
            "sample_supported": False,
            "archive_wide_guarantee": False,
            "engineering_decision": (
                "Individual pointing structure not "
                "available from tested ObsCore fields"
            ),
        },
        {
            "entity": "FormalDuplicationDecision",
            "sample_supported": False,
            "archive_wide_guarantee": False,
            "engineering_decision": (
                "Requires Queue CSV and known-case tests"
            ),
        },
    ]
)

display(notebook04_final_readiness_df)

,entity,sample_supported,archive_wide_guarantee,engineering_decision
0,ArchiveRawRow,True,False,Preserve all selected raw fields
1,ObsIdStructure,True,False,Parse with status and preserve raw obs_id
2,SourceExecutionContext,True,False,Member + ASDM + source context
3,SourceSpwRecord,True,False,Execution context + SPW identifier
4,FrequencySupportComponent,True,False,"Preserve raw, parsed values, status, issues an..."
5,ObservationModeEvidence,True,False,Store bandwidth/resolution evidence without as...
6,ObservationModeClassification,False,False,Remain UNKNOWN until policy evidence is defined
7,SpatialFootprint,True,False,Preserve aggregate STC-S region
8,MosaicPointing,False,False,Individual pointing structure not available fr...
9,FormalDuplicationDecision,False,False,Requires Queue CSV and known-case tests


## 11. Final Conclusions

1. The corrected sample contained 1,619 complete science-target Archive rows
   from 19 Member OUS datasets and 20 ASDM executions.

2. All tested `obs_id` values were parsed into Member OUS, source, and SPW
   components. The raw `obs_id` must nevertheless remain preserved because
   the observed grammar is not an Archive-wide contract.

3. A conservative Source Execution Context is identified by Member OUS, ASDM,
   and source name. A Source-SPW record additionally contains the parsed SPW
   identifier.

4. The production frequency-support parser successfully parsed all 1,619
   components without parse or validation issues.

5. All 376 tested Source Execution Contexts contained equal Archive-row and
   parsed-component counts.

6. Simple interval containment was ambiguous for 226 rows because frequency
   intervals can overlap. Context-level one-to-one frequency assignment
   resolved all 1,619 rows without count mismatches or out-of-interval
   assignments.

7. Archive `bandwidth` and parsed interval width are related but not
   numerically interchangeable. They nevertheless agreed on the 1.8 GHz
   threshold for every tested row.

8. Archive `spectral_resolution` closely matched parsed component resolution
   and is supported as SPW-level evidence.

9. Archive `em_resolution` closely matched the wavelength-resolution value
   derived from frequency and component resolution. It should be treated as
   derived metadata.

10. Archive `velocity_resolution` was constant within each tested Member OUS
    but did not consistently match row-level velocity resolution derived from
    frequency and component resolution. It must not replace SPW-level spectral
    resolution.

11. Archive `sensitivity_10kms`, parsed `@10km/s` sensitivity, and parsed
    `@native` sensitivity vary at SPW/component level. Both raw and parsed
    representations must be preserved.

12. `cont_sensitivity_bandwidth` was constant within each Source Execution
    Context and is supported as an aggregate source-context sensitivity. It is
    not equivalent to component-level native sensitivity.

13. ObsCore provided one coordinate and one aggregate STC-S footprint per
    Source Execution Context. Individual mosaic pointings were not demonstrated
    and cannot currently be reconstructed from the tested fields.

14. No direct FDM/TDM observation-mode field was found. The implementation may
    store observation-mode evidence, but formal mode classification must remain
    `UNKNOWN` until an authoritative rule or additional metadata source is
    established.

15. The results support implementation of raw Archive records, observation
    hierarchy, Source Execution Contexts, Source-SPW records, parsed frequency
    components, spatial footprints, and traceable evidence objects. Formal
    duplication decisions require Queue CSV exploration and known-case
    validation.